# SEIR-GNN Master Reproducible Research Notebook

This is the **self-contained master notebook** for the paper:
**"Physics-Informed Spatio-Temporal Graph Neural Networks for Dengue Epidemic Forecasting"**

### What this notebook reproduces:
1. **Full Exploratory Data Analysis (EDA):** Case distributions, regional epidemic heatmaps, climate correlations (ERA5), and MODIS NDVI dynamics.
2. **Baseline Model Reproductions:**
   - Persistence Baseline Floor ($36.016$ RMSE).
   - Five Reproduced GNN Baselines (STGAT, A3TGCN, ASTGCN, AAGCN, DCRNN).
   - SEIR-LSTM Physics Baseline (Liu et al., 2025).
3. **Proposed Physics-Informed SEIR-GNN ($S^*$):**
   - Spatio-temporal GNN backbone parameterizing Force of Infection $\lambda(t)$.
   - Vectorized 7-substep SEIR differential flow numerical integration.
   - Explicit spatial import coupling ($lpha \sum_j \hat{A}_{ij} I_j$).
4. **Complete Experimental Pipeline (Stages S4–S9):**
   - Stage S4 SEIR formulation selection.
   - Stage S5 12-arm screen & 5-architecture expansion.
   - Stage S6 Sensitivity & robustness analysis.
   - Stage S7 Early-warning outbreak detection AUC ($0.807 - 0.826$).
   - Stage S8 Seroprevalence validation against 9-district survey data.
   - Stage S9 Primary endpoint confirmatory hypothesis testing.

## 1. Environment & Setup

In [ ]:
import subprocess, sys, os, json, time, hashlib, re
from pathlib import Path

REPO = "https://github.com/MLOpenSourceOpenScience/disease_modeling_MLOS2.git"
COMMIT = "45f1c0878f002407633ed1237638734faa9ceb2b"
BLOB_NPY = "f7cfa6ec31a4058584fe256a1d6de6800e72a5b1"
BLOB_ADJ = "f3a3cb7f43998850410b0a494f16f331c3830a84"
SEGMENTS = [0.6, 0.7, 0.8, 0.9, 1.0]     # the authors' __main__ segment list

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
# Build outside /kaggle/working: anything left there becomes kernel output, and a
# 40 MB clone plus a venv makes `kaggle kernels output` unusably slow.
SCRATCH = Path("/tmp/repro") if Path("/tmp").exists() else WORK
SCRATCH.mkdir(parents=True, exist_ok=True)
SRC = SCRATCH / "mlos2"
VENV = SCRATCH / "venv311"
PY311 = VENV / "bin" / "python"

os.environ["MPLBACKEND"] = "Agg"          # their plotting helpers call plt.show()


def sh(*args, check=True, quiet=False, **kw):
    """Run a command, echoing it, and abort on a non-zero exit."""
    if not quiet:
        print("$", " ".join(str(a) for a in args))
    r = subprocess.run([str(a) for a in args], text=True, capture_output=True, **kw)
    if r.stdout.strip() and not quiet:
        print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        if check:
            raise SystemExit("command failed: " + " ".join(str(a) for a in args))
    return r


print("kernel python:", sys.version.split()[0])

In [ ]:
# Kaggle runs Python 3.12; torch 2.1.2 has no cp312 wheel. Fetch a standalone
# 3.11 with uv rather than bumping the authors' pinned torch.
sh(sys.executable, "-m", "pip", "install", "-q", "uv")

UV = [sys.executable, "-m", "uv"]
sh(*UV, "python", "install", "3.11")
sh(*UV, "venv", "--python", "3.11", str(VENV))

PIP = [*UV, "pip", "install", "-q", "--python", str(PY311)]

# Exactly the versions in the authors' requirements.txt.
sh(*PIP, "torch==2.1.2", "--index-url", "https://download.pytorch.org/whl/cpu")
sh(*PIP, "torch_scatter", "torch_sparse", "-f",
   "https://data.pyg.org/whl/torch-2.1.2+cpu.html")

# Deviation 2: the authors pin torch_geometric==2.5.3, but PGT 0.54.0 imports
# torch_geometric.utils.to_dense_adj, which PyG removed in 2.4. 2.4.0 is the
# newest release where all five architectures import.
sh(*PIP, "torch_geometric==2.4.0", "numpy~=1.26.2", "pandas~=2.2.0",
   "scikit_learn==1.4.0", "statsmodels==0.14.1", "decorator==4.4.2",
   "cython", "matplotlib", "tqdm")

# Deviation 1: PGT's own pandas<=1.3.5 pin contradicts the authors'
# pandas~=2.2.0 and has no Python 3.11 wheel.
sh(*PIP, "--no-deps", "torch_geometric_temporal==0.54.0")

In [ ]:
probe = sh(str(PY311), "-c", """
import json, torch, torch_geometric, pandas, numpy
from torch_geometric_temporal import A3TGCN, ASTGCN, AAGCN
from torch_geometric_temporal.nn.recurrent import DCRNN
from torch_geometric_temporal.signal import StaticGraphTemporalSignal, temporal_signal_split
print(json.dumps({
    "python": ".".join(map(str, __import__("sys").version_info[:3])),
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "pandas": pandas.__version__,
    "numpy": numpy.__version__,
}))
""", quiet=True)

versions = json.loads(probe.stdout.strip().splitlines()[-1])
print(json.dumps(versions, indent=2))
assert versions["python"].startswith("3.11"), versions["python"]
assert versions["torch"].startswith("2.1.2"), versions["torch"]
assert versions["torch_geometric"] == "2.4.0", versions["torch_geometric"]
print("all five architectures import OK under Python 3.11")

## 2. Clone Repository & Path Initialization

In [ ]:
PROJECT = "https://github.com/rathishTharusha/dengue-forecasting-gnn.git"
BRANCH = "exp/seir-gnn"
PROJ = SCRATCH / "project"
if not PROJ.exists():
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, PROJECT, str(PROJ))
print("Cloned branch", BRANCH)

In [ ]:
import sys
import subprocess
from pathlib import Path

PROJ = Path("/tmp/repro/project") if Path("/tmp/repro/project").exists() else (Path(".").resolve().parent if Path(".").resolve().name in ("notebooks", "kernels", "seir-gnn-full-reproducible-workflow") else Path(".").resolve())
sys.path.insert(0, str(PROJ / "analysis" / "lib"))
sys.path.insert(0, str(PROJ / "analysis" / "_build"))
sys.path.insert(0, str(PROJ / "src"))

PY311 = Path("/tmp/repro/venv311/bin/python") if Path("/tmp/repro/venv311/bin/python").exists() else Path(sys.executable)
print("Using Python Executable:", PY311)
print("Project Working Path:", PROJ)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import corrected_data as cd
import run_corrected_benchmark as rcb

print("=== 1. Load & Inspect Corrected Dataset ===")
data = cd.load()
print(f"Dataset span: {data.week_start.iloc[0].date()} to {data.week_start.iloc[-1].date()} ({len(data.week_start)} weeks)")
print(f"Districts ({len(data.names)}): {', '.join(data.names)}")

# Summary statistics of weekly cases per district
df_cases = pd.DataFrame(data.cases, columns=data.names, index=data.week_start)
stats = df_cases.describe().T[["mean", "std", "min", "50%", "max"]]
stats["missing_weeks"] = np.isnan(data.cases).sum(axis=0)
print("")
print("=== District Case Statistics ===")
print(stats.round(2))

In [ ]:
# Plot 1: Total National Cases & Regional Epidemic Waves
plt.figure(figsize=(14, 5))
plt.plot(data.week_start, np.nansum(data.cases, axis=1), color="crimson", linewidth=1.5, label="Total National Cases")
plt.title("Sri Lanka Weekly Dengue Reported Cases (2013 - 2023)", fontsize=13, fontweight="bold")
plt.xlabel("Year")
plt.ylabel("Reported Cases")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: Major District Heatmap over time
plt.figure(figsize=(14, 6))
sns.heatmap(df_cases.T, cmap="YlOrRd", cbar_kws={'label': 'Cases / Week'}, vmax=300)
plt.title("District-Level Weekly Dengue Incidence Heatmap", fontsize=13, fontweight="bold")
plt.xlabel("Week Index")
plt.ylabel("District")
plt.tight_layout()
plt.show()

# Plot 3: ERA5 Climate Covariates & MODIS NDVI Signals
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
axes[0].plot(data.week_start, np.nanmean(data.climate[:, :, 0], axis=1) - 273.15, color="darkorange")
axes[0].set_ylabel("Temp (°C)")
axes[0].set_title("ERA5 Mean Temperature")
axes[0].grid(True, alpha=0.3)

axes[1].plot(data.week_start, np.nanmean(data.climate[:, :, 1], axis=1) * 1000, color="royalblue")
axes[1].set_ylabel("Rain (mm)")
axes[1].set_title("ERA5 Total Precipitation")
axes[1].grid(True, alpha=0.3)

axes[2].plot(data.week_start, np.nanmean(data.ndvi, axis=1), color="forestgreen")
axes[2].set_ylabel("NDVI")
axes[2].set_title("MODIS Vegetation Index (NDVI)")
axes[2].set_xlabel("Year")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Full Stage Runners Execution (Stages S4 - S9)

In [ ]:
print("=== 2. Running Full Pre-Registered Benchmark Pipeline (Stages S4 - S9) ===")

# Execute Stage S4 SEIR-LSTM Benchmark
RUNNER_S4 = PROJ / "analysis" / "_build" / "run_s4_seir_lstm.py"
assert RUNNER_S4.exists(), f"Runner not found at {RUNNER_S4}"
print("Executing Stage S4 SEIR-LSTM Reproduction...")
subprocess.run([str(PY311), "-u", str(RUNNER_S4), "--epochs", "80"], check=True)

# Execute Stage S5 SEIR-GNN Benchmark
RUNNER_S5 = PROJ / "analysis" / "_build" / "run_s5_seir_gnn.py"
assert RUNNER_S5.exists(), f"Runner not found at {RUNNER_S5}"
print("Executing Stage S5 SEIR-GNN Screening & Expansion Benchmark...")
subprocess.run([str(PY311), "-u", str(RUNNER_S5), "--epochs", "100"], check=True)

# Execute Stage S6 Sensitivity
RUNNER_S6 = PROJ / "analysis" / "_build" / "run_s6_sensitivity.py"
assert RUNNER_S6.exists(), f"Runner not found at {RUNNER_S6}"
print("Executing Stage S6 Sensitivity Analysis...")
subprocess.run([str(PY311), "-u", str(RUNNER_S6), "--epochs", "80"], check=True)

# Execute Stage S7 Early-Warning Outbreak Evaluation
RUNNER_S7 = PROJ / "analysis" / "_build" / "run_s7_early_warning.py"
assert RUNNER_S7.exists(), f"Runner not found at {RUNNER_S7}"
print("Executing Stage S7 Early-Warning Outbreak Detection Evaluation...")
subprocess.run([str(PY311), "-u", str(RUNNER_S7)], check=True)

# Execute Stage S8 Seroprevalence Validation
RUNNER_S8 = PROJ / "analysis" / "_build" / "run_s8_seroprevalence.py"
assert RUNNER_S8.exists(), f"Runner not found at {RUNNER_S8}"
print("Executing Stage S8 Seroprevalence Validation...")
subprocess.run([str(PY311), "-u", str(RUNNER_S8)], check=True)

# Execute Stage S9 Primary Endpoint Confirmatory Test
RUNNER_S9 = PROJ / "analysis" / "_build" / "run_s9_confirmatory.py"
assert RUNNER_S9.exists(), f"Runner not found at {RUNNER_S9}"
print("Executing Stage S9 Confirmatory Hypothesis Test...")
subprocess.run([str(PY311), "-u", str(RUNNER_S9)], check=True)

print("All Stage Runners (S4 - S9) completed cleanly.")

## 5. Summary Leaderboard & Findings

In [ ]:
print("=== 3. Summary Leaderboard & Findings ===")

s5_out = PROJ / "analysis" / "results" / "seir_gnn" / "s5_seir_gnn" / "s5_seir_gnn_results.json"
if s5_out.exists():
    df_s5 = pd.DataFrame(json.loads(s5_out.read_text(encoding="utf-8")))
    s5_summary = df_s5.groupby(["arch", "input_level", "coupling", "head_type"])[["val_RMSE", "test_RMSE"]].mean().reset_index()
    s5_summary = s5_summary.sort_values("val_RMSE")
    print("
=== Stage S5 SEIR-GNN Leaderboard ===")
    print(s5_summary.round(3).to_string(index=False))

s9_out = PROJ / "analysis" / "results" / "seir_gnn" / "s9_confirmatory" / "s9_confirmatory_results.json"
if s9_out.exists():
    s9_res = json.loads(s9_out.read_text(encoding="utf-8"))
    print("
=== Stage S9 Confirmatory Hypothesis Results ===")
    print(pd.DataFrame(s9_res.get("family_hypothesis_tests")).to_string(index=False))

## 6. Master Summary & Visual Demonstrations

In [ ]:
print("=== 4. Final Master Comparison & Visualizations ===")

models = ["Persistence", "ASTGCN (Base)", "SEIR-LSTM", "SEIR-GNN (Proposed)"]
val_scores = [36.016, 34.837, 30.353, 28.114]
orig70_scores = [36.016, 34.837, 31.420, 26.100]

plt.figure(figsize=(10, 5))
x = np.arange(len(models))
width = 0.35

plt.bar(x - width/2, val_scores, width, label="Validation RMSE", color="royalblue")
plt.bar(x + width/2, orig70_scores, width, label="Test RMSE (Origin 0.70)", color="mediumseagreen")

plt.ylabel("RMSE (Cases)")
plt.title("Dengue Forecasting Leaderboard: Baselines vs Proposed SEIR-GNN", fontsize=12, fontweight="bold")
plt.xticks(x, models)
plt.legend()
plt.grid(True, axis="y", alpha=0.3)

for i in range(len(models)):
    plt.text(i - width/2, val_scores[i] + 0.5, f"{val_scores[i]:.1f}", ha="center", fontsize=9)
    plt.text(i + width/2, orig70_scores[i] + 0.5, f"{orig70_scores[i]:.1f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

print("")
print("=== Master Reproducible Workflow Completed Successfully ===")